# Qwen2.5-3B-Instruct 中文聊天 SFT（QLoRA）— Colab T4 训练（不用 Drive）

基础模型：Qwen/Qwen2.5-3B-Instruct · 数据：NaturalConv 3500 条中文多轮对话（ShareGPT 格式）· 方法：4-bit QLoRA

**不用 Google Drive**：数据文件在第 3 格运行时直接上传到 Colab；训练约 30–60 分钟，跑完 adapter 打包成 zip 自动下载。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 安装 LlamaFactory（约 5–10 分钟，只需跑一次）
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git /content/LlamaFactory
!pip install -q -e "/content/LlamaFactory[torch,metrics,bitsandbytes]"

In [ ]:
# 直接上传数据文件到 Colab（不用 Drive）
# 运行这格后会出现上传框，选择 train.json 和 validation.json 两个文件
from google.colab import files
import json, shutil
from pathlib import Path

uploaded = files.upload()  # 在上传框里选 train.json 和 validation.json
print('uploaded:', list(uploaded.keys()))

LF_DATA = Path('/content/LlamaFactory/data')
for name in ['train.json', 'validation.json']:
    src = Path('/content') / name
    assert src.exists(), f'没找到 {name}，请在上传框里选择它'
    shutil.copy(src, LF_DATA / name)
    print('copied', name)

dataset_info = {
    'naturalconv_train': {'file_name': 'train.json', 'formatting': 'sharegpt', 'columns': {'messages': 'messages'}},
    'naturalconv_valid': {'file_name': 'validation.json', 'formatting': 'sharegpt', 'columns': {'messages': 'messages'}},
}
(LF_DATA / 'dataset_info.json').write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2))
print('dataset_info.json written')


In [ ]:
from pathlib import Path
yaml_text = '# Qwen2.5-3B-Instruct 中文聊天 SFT —— QLoRA 配置（单张 T4 16GB）\n# 用法：在 LlamaFactory 仓库根目录运行：llamafactory-cli train llamafactory_qlora.yaml\n# 数据文件 train.json / validation.json + dataset_info.json 放在 LlamaFactory 的 data/ 目录下\n\n### 模型\nmodel_name_or_path: Qwen/Qwen2.5-3B-Instruct\ntrust_remote_code: true   # Qwen 系模型需要\n\n### 方法：QLoRA（4-bit 量化底座 + 只训 LoRA 适配器）\nstage: sft\ndo_train: true\nfinetuning_type: lora\nlora_target: all          # 训所有线性层（q/k/v/o + gate/up/down），聊天风格学得更全\nlora_rank: 16             # rank 16：3B 模型 + 3500 条数据够用；再大容易过拟合且更占显存\nlora_alpha: 32            # alpha = 2×rank，标准配比，缩放因子 2\nlora_dropout: 0.05        # 轻微 dropout 防过拟合\nquantization_bit: 4       # 4-bit NF4 量化底座：3B 模型只占 ~2GB 显存，T4 塞得下\nquantization_method: bitsandbytes\n\n### 数据\ndataset: naturalconv_train\neval_dataset: naturalconv_valid\ndataset_dir: data\ntemplate: qwen2           # Qwen2.5-Instruct 的官方 chat 模板\ncutoff_len: 1024          # 对话平均 ~260 token，1024 覆盖全部样本且不浪费\npacking: false            # 不打包：多轮对话逐条训，loss 只算 assistant 回复，质量更稳\noverwrite_cache: true\npreprocessing_num_workers: 16\n\n### 训练超参\nper_device_train_batch_size: 4      # 每步 4 条；配合梯度累积，有效 batch 16\ngradient_accumulation_steps: 4      # 有效 batch = 4×4 = 16，3500 条 ≈ 219 步/epoch\nlearning_rate: 1.0e-4               # QLoRA 常用 lr；LoRA 只训少量参数，lr 可以比全参高\nnum_train_epochs: 3.0               # 3 epoch：3500 条小数据，3 轮够学风格，更多易过拟合\nlr_scheduler_type: cosine           # 余弦衰减，收尾更稳\nwarmup_ratio: 0.03                  # 前 3% 步数热身，避免初期震荡\nbf16: true                          # T4 支持 bf16，比 fp16 更稳\ngradient_checkpointing: true        # 用少量速度换显存：激活不全存，T4 必开\noptim: paged_adamw_8bit             # 8-bit 分页 AdamW：优化器状态只占 ~50MB（QLoRA 标配）\nmax_grad_norm: 1.0                  # 梯度裁剪防爆炸\n\n### 输出与评估\noutput_dir: saves/Qwen2.5-3B-Instruct/lora/naturalconv-sft\nlogging_steps: 10\nsave_steps: 200                     # 每 200 步存一个 checkpoint（3 epoch 共 ~656 步）\neval_strategy: steps\neval_steps: 200\nper_device_eval_batch_size: 4\nplot_loss: true\noverwrite_output_dir: true\nreport_to: none                     # 不用 wandb，不需要 token\n'
Path('/content/train_qlora.yaml').write_text(yaml_text)
print('train_qlora.yaml written')

In [ ]:
# 开始训练（约 30–60 分钟，约 656 个优化步；保持标签页打开）
!cd /content/LlamaFactory && llamafactory-cli train /content/train_qlora.yaml

In [ ]:
# 把训好的 adapter 打包成 zip 并下载（约 50MB，不用 Drive）
import shutil
from pathlib import Path
from google.colab import files

out = Path('/content/LlamaFactory/saves/Qwen2.5-3B-Instruct/lora/naturalconv-sft')
assert out.exists(), f'没找到 adapter 目录 {out}'
zip_path = shutil.make_archive('/content/naturalconv-adapter', 'zip', out)
print('adapter zipped:', zip_path)
files.download(zip_path)


In [ ]:
# 快速验证：加载 adapter，聊两句
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct', load_in_4bit=True, device_map='auto', torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, '/content/LlamaFactory/saves/Qwen2.5-3B-Instruct/lora/naturalconv-sft')
model.eval()

messages = [{'role': 'user', 'content': '你好啊，今天过得怎么样？'}]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(text, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.7, top_p=0.9)
print(tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))


## 注意事项

- 如果 Colab 中途断开：重新跑上面各格，在训练前把 yaml 里的 `output_dir` 保持不变，并加上 `resume_from_checkpoint: true` 即可从最近的 checkpoint 续训（checkpoint 在 Colab 运行时里，断开后会丢失，续训只对同一次运行时有效）。
- adapter 约 50MB，已打包成 naturalconv-adapter.zip 下载。
- 想要合并成完整模型再导出 GGUF：另开一个 notebook，用 LlamaFactory 的 export 功能（`llamafactory-cli export`），合并需要约 12GB 内存，Colab 免费版 CPU 内存够用。
